<div style="display:flex; justify-content:space-between; align-items:center; width:100%; margin:8px 0 24px 0;">
  <div style="text-align:left;">
    <img src="https://www.ec-nantes.fr/medias/photo/logocn-rvb_1648479844750-png?ID_FICHE=178994&amp;INLINE=FALSE" alt="Centrale Nantes" style="height:72px; width:auto;">
  </div>
  <div style="text-align:right; font-size:18px; font-weight:600; color:#17324d; line-height:1.35;">
    MSc. CORO DASSIP
  </div>
</div>

<div style="border:2px solid #333; padding:14px 20px; margin:15px auto 25px auto; width:85%; max-width:900px; box-sizing:border-box; text-align:center;">
  <h1 style="margin:0;"><b>Image Segmentation</b></h1>
</div>

## Context

Image segmentation converts pixel-level evidence into discrete object or region labels. Classical segmentation quality depends on the separability of intensity/color distributions, illumination uniformity, object connectivity, boundary definition, morphological structure, and the choice of evaluation metric.

The laboratory examines complementary segmentation mechanisms rather than assuming a single universal method. Thresholding, morphology, region analysis, color cues, distance transforms, and watershed are evaluated according to the image conditions they are intended to address.

## Problem Statement

Implement and evaluate a classical image-segmentation pipeline that produces interpretable binary or labeled regions under varying image conditions.

The work must:

- characterize image histograms and threshold sensitivity;
- compare manual, Otsu, and adaptive thresholding;
- evaluate Gaussian preprocessing before thresholding;
- implement erosion, dilation, opening, closing, morphological gradient, and hole filling;
- extract connected components, contours, and region properties;
- remove components using explicit geometric criteria;
- perform color-based segmentation in HSV space;
- investigate edge-driven, distance-transform, and watershed strategies;
- evaluate segmentation using pixel accuracy, precision, recall, IoU, and Dice when ground truth is available;
- distinguish under-segmentation from over-segmentation;
- validate mask semantics, region consistency, metrics, and generated figures.

Deep semantic segmentation and learned instance-segmentation models are outside scope.

## Inputs and Fixed Parameters

| Item | Fixed value / convention |
| --- | --- |
| Data directory | `../data/` |
| Reference images | `hand.png`, `peppers.png`, `tower.jpg` |
| Primary libraries | NumPy, SciPy, OpenCV, Pillow, Matplotlib |
| Binary-mask convention | explicitly stated for every experiment |
| Global threshold methods | manual, Otsu |
| Local thresholding | adaptive mean / Gaussian where applicable |
| Morphology | erosion, dilation, opening, closing, gradient, hole filling |
| Region analysis | connected components, contours, geometric properties |
| Color space | HSV for color-range experiments |
| Separation method | distance transform + watershed |
| Evaluation | pixel accuracy, precision, recall, IoU, Dice |
| Output directory | `../outputs/figures/` |
| Required figures | 21 named diagnostics |

All mask operations must preserve binary semantics. Interpolation that creates non-binary mask values is not permitted unless followed by an explicitly justified re-thresholding step.

## 1. Segmentation Problem Formulation

Define the segmentation output as a binary mask

$$
M(x,y)=
\begin{cases}
1,&\text{foreground}\\
0,&\text{background}.
\end{cases}
$$

For each supplied image selected for an experiment:

1. state what is considered foreground;
2. state which observable cue is expected to separate it from background;
3. identify likely failure modes such as uneven illumination, weak contrast, touching objects, or textured background;
4. define the intended output representation.

No segmentation result may be evaluated without an explicit foreground/background convention.

## 2. Load the Lab Images

Load `hand.png`, `peppers.png`, and `tower.jpg`.

For each image, report:

- shape;
- channel count;
- dtype;
- intensity range;
- grayscale or color representation used in later tasks.

Save a common input montage as `01_input_images.png`.

Reject unreadable inputs explicitly.

## 3. Histograms and Why Thresholding Can Work

For a selected grayscale image:

1. compute a 256-bin intensity histogram;
2. identify candidate foreground/background modes;
3. propose one or more threshold ranges from the histogram;
4. relate the histogram structure to expected segmentation difficulty.

Save `02_hand_histogram.png`.

Thresholding is justified only when the intensity distribution provides separable or partially separable classes.

## 4. Manual Global Thresholding

Implement

$$
M_T(x,y)=
\mathbf 1[I(x,y)>T]
$$

or the complementary convention when the foreground is darker.

Choose at least three manual threshold values around the histogram-based candidate.

Save `03_manual_threshold.png`.

For each threshold, report foreground-pixel fraction and visually inspect missing foreground versus false background inclusion.

## 5. Threshold Sensitivity

Evaluate a controlled sweep of threshold values while keeping all preprocessing fixed.

For each threshold $T$, record at minimum:

- foreground area;
- number of connected regions;
- segmentation metric(s) if ground truth is available.

Save `04_threshold_sensitivity.png`.

Identify whether the segmentation is stable over a useful interval or depends on a narrow operating point.

## 6. Otsu Thresholding

Apply Otsu's global threshold selection.

Let class probabilities and means induced by threshold $T$ be used to maximize between-class separation, equivalently minimizing within-class variance.

Report:

- Otsu threshold value;
- foreground fraction;
- difference from the manually selected operating range.

Save `05_otsu_threshold.png`.

Discuss whether the histogram is sufficiently bimodal for the Otsu assumption to be appropriate.

## 7. Gaussian Smoothing Before Thresholding

Apply Gaussian smoothing before global thresholding while holding the segmentation rule fixed.

Evaluate at least two smoothing scales.

Save `06_smoothing_before_otsu.png`.

Compare:

- threshold value;
- number of isolated mask components;
- boundary sharpness;
- overlap metrics where ground truth is available.

Determine whether smoothing suppresses noise or erases relevant thin structures.

## 8. Adaptive Thresholding

Apply local adaptive thresholding to an image with spatially varying illumination.

Evaluate at least:

- adaptive mean;
- adaptive Gaussian.

Document neighborhood/block size and offset constant.

Save `07_adaptive_thresholding.png`.

Compare adaptive results with the best global threshold under identical input preprocessing.

## 9. Morphological Processing

Starting from one noisy binary mask, evaluate how morphology changes object support.

Apply controlled erosion/dilation-based cleanup and record:

- foreground area before/after;
- connected-component count before/after;
- whether narrow structures are removed or merged.

The original mask must be preserved for comparison.

## 10. Structuring Elements

Construct at least three structuring-element shapes:

- rectangle;
- ellipse/disk;
- cross.

Evaluate more than one size.

Save `08_structuring_elements.png`.

For each element, report shape, dimensions, and the type of geometric structure it preferentially preserves or removes.

## 11. Erosion and Dilation

Apply erosion and dilation separately to the same binary mask.

For set $A$ and structuring element $B$, implement the standard operations $A\ominus B$ and $A\oplus B$.

Save `09_erosion_dilation.png`.

Quantify the change in foreground area and component connectivity after each operation.

## 12. Opening and Closing

Implement

$$
A\circ B=(A\ominus B)\oplus B
$$

and

$$
A\bullet B=(A\oplus B)\ominus B.
$$

Apply both to the same initial mask using documented structuring elements.

Save `10_opening_closing.png`.

Measure changes in small-component count and internal holes.

## 13. Morphological Gradient

Compute

$$
G=(A\oplus B)-(A\ominus B)
$$

for a selected binary mask.

Save `11_morphological_gradient.png`.

Compare the resulting boundary band with a contour representation and report its approximate thickness relative to the structuring-element size.

## 14. Hole Filling

Select or generate a binary mask containing internal holes.

Apply binary hole filling.

Save `12_hole_filling.png`.

Report:

- number of holes removed;
- foreground-area increase;
- confirmation that exterior background is not incorrectly filled.

## 15. Connected Components

Label the connected foreground regions.

For each component, compute at least:

- label;
- area;
- centroid;
- bounding box.

Save `13_connected_components.png`.

Report the total component count and verify that background uses a distinct label.

## 16. Remove Small Components

Define a minimum area $A_{min}$ and retain only components satisfying

$$
A_k\ge A_{min}.
$$

Evaluate multiple $A_{min}$ values.

Save `14_component_area_filtering.png`.

Report the number of removed/retained components and verify that the selected threshold does not delete the principal target object.

## 17. Contours

Extract ordered object contours from a cleaned binary mask.

For representative contours, compute:

- perimeter;
- area;
- bounding rectangle.

Save `15_contours.png`.

Verify that contour geometry is consistent with the corresponding connected-component region.

## 18. Region Properties

For selected segmented regions, compute:

- area $A$;
- perimeter $P$;
- centroid;
- aspect ratio;
- circularity

$$
C=
\frac{4\pi A}{P^2}.
$$

Present the measurements in a table.

Check that circularity lies in a physically meaningful range and interpret deviations for non-circular objects.

## 19. Color Segmentation

Convert `peppers.png` or another color reference image to HSV.

1. inspect hue, saturation, and value channels;
2. define one target color range;
3. create a binary mask;
4. optionally refine it morphologically.

Save:

- `16_hsv_channels.png`;
- `17_color_segmentation.png`.

Report the exact HSV bounds used and the resulting foreground fraction.

## 20. Edge-Based Segmentation Intuition

Construct an edge-driven segmentation experiment:

1. compute an edge map using a stated detector;
2. connect gaps using a controlled morphological operation;
3. fill enclosed regions where appropriate;
4. obtain a candidate binary mask.

Save `18_edge_based_segmentation.png`.

Compare the mask with an intensity-based method and identify failure caused by broken or spurious edges.

## 21. Distance Transform

For a binary foreground mask, compute the distance transform

$$
D(x,y)
=
\min_{(u,v)\in\text{background}}
d((x,y),(u,v)).
$$

Save `19_distance_transform.png`.

Identify local maxima associated with object centers and use a thresholded distance map to construct candidate sure-foreground markers.

## 22. Watershed Segmentation

Implement a marker-based watershed pipeline:

1. obtain an initial binary foreground mask;
2. remove isolated noise;
3. determine sure background;
4. compute distance transform;
5. determine sure foreground;
6. construct the unknown region;
7. label markers;
8. apply watershed.

Save `20_watershed.png`.

Report the number of final regions and inspect whether touching objects are successfully separated without severe over-segmentation.

## 23. Ground Truth and Segmentation Metrics

For predicted mask $M$ and reference mask $G$, compute $TP$, $TN$, $FP$, and $FN$.

Then compute:

$$
\mathrm{Accuracy}
=
\frac{TP+TN}{TP+TN+FP+FN},
$$

$$
\mathrm{Precision}
=
\frac{TP}{TP+FP},
$$

$$
\mathrm{Recall}
=
\frac{TP}{TP+FN},
$$

$$
\mathrm{IoU}
=
\frac{TP}{TP+FP+FN},
$$

$$
\mathrm{Dice}
=
\frac{2TP}{2TP+FP+FN}.
$$

Handle zero denominators explicitly and verify mask shape/convention before comparison.

## 24. Dice vs IoU Relationship

Verify numerically that

$$
\mathrm{Dice}
=
\frac{2\mathrm{IoU}}
{1+\mathrm{IoU}}
$$

and

$$
\mathrm{IoU}
=
\frac{\mathrm{Dice}}
{2-\mathrm{Dice}}.
$$

Use multiple predicted masks with different overlap quality.

Report the numerical residual of each identity and confirm that both metrics preserve the same ranking for the tested predictions.

## 25. Under-Segmentation vs Over-Segmentation

Construct or identify examples of:

- under-segmentation: distinct objects merged;
- over-segmentation: one object split into multiple regions.

For each example, report:

- connected-component count;
- qualitative cause;
- whether pixel-overlap metrics alone fully describe the region-partition error.

Relate these cases to the watershed experiment.

## 26. End-to-End Binary Segmentation Pipeline

Implement one reusable classical segmentation pipeline containing, as justified by the selected input:

1. grayscale or color-space conversion;
2. noise reduction;
3. thresholding;
4. morphological cleanup;
5. hole filling;
6. component filtering;
7. contour/region extraction;
8. final mask and overlay generation.

Save `21_complete_pipeline.png`.

Report the exact retained parameters and final metrics.

## 27. Segmentation Method Selection Criteria

Using results from the laboratory, construct a method-selection table for at least these conditions:

- strong global intensity separation;
- uneven illumination;
- color-dominant separation;
- noisy binary masks;
- touching objects;
- boundary-driven objects.

For each condition, identify the justified method and the evidence from the corresponding experiment.

Do not select methods solely by algorithmic complexity.

## 28. Integrated Segmentation Workflow

Apply the full decision process to one supplied image from raw input to evaluated mask.

Document:

- observed image characteristics;
- selected segmentation strategy;
- preprocessing parameters;
- post-processing parameters;
- region-analysis settings;
- final metrics;
- principal failure modes.

The resulting workflow must be reproducible from the reported configuration without manual tuning during execution.

## Completion Criterion

The laboratory is complete when global/local thresholding, morphology, component/contour analysis, color segmentation, distance-transform/watershed separation, and quantitative mask evaluation have all been exercised under explicit conventions, the end-to-end pipeline is reproducible, and all 21 required diagnostic figures and validation results are available.